# Notebook 3: World Map — Data Preparation & Static Plots

This notebook prepares the per-country summary data used by the `/map` page in the web app,
and generates a few static choropleth-style overview plots saved to `app/static/`.

The web app itself renders an interactive map using D3.js + GeoJSON — this notebook only
handles the data side and generates the static backup figures.

**Input:** `data/processed_data.csv`  
**Outputs:**
- `app/static/map_data.json` — per-country, per-year summary for the interactive map
- `app/static/map_energy_2021.png` — static choropleth (energy consumption)
- `app/static/map_renewables_2021.png` — static choropleth (renewables share)

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.colorbar import ColorbarBase
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13
})

df = pd.read_csv('../data/processed_data.csv')
print(f"Loaded: {df.shape}")
df.head(3)

## 1. Build Per-Country Summary JSON

For the interactive map we need a lookup by ISO code + year.
We compute a few extra derived fields that are useful for the map tooltips.

In [ ]:
# Derived fields
df['gdp_per_capita'] = (df['gdp'] / df['population']).round(0)
df['energy_per_capita_kwh'] = (df['energy_per_capita'] * 1000).round(0)  # convert MWh → kWh label

# Choose which fields go into the map JSON
MAP_FIELDS = [
    'country', 'iso_code', 'year',
    'primary_energy_consumption',
    'energy_per_capita',
    'renewables_share_elec',
    'fossil_share_elec',
    'solar_share_elec',
    'wind_share_elec',
    'hydro_share_elec',
    'nuclear_share_elec',
    'electricity_generation',
    'coal_production',
    'oil_production',
    'gas_production',
    'gdp_per_capita',
    'population'
]

map_df = df[MAP_FIELDS].copy()

# Round to keep JSON small
float_cols = map_df.select_dtypes(include='float').columns
map_df[float_cols] = map_df[float_cols].round(2)

print(f"Map dataset shape: {map_df.shape}")
map_df.head(3)

In [ ]:
# Build nested JSON: { year: { iso_code: { ...fields } } }
# This lets the JS side do O(1) lookup per country per year

map_json = {}

for year, year_df in map_df.groupby('year'):
    map_json[int(year)] = {}
    for _, row in year_df.iterrows():
        iso = row['iso_code']
        map_json[int(year)][iso] = {
            'country': row['country'],
            'energy': row['primary_energy_consumption'],
            'energy_per_capita': row['energy_per_capita'],
            'renewables_pct': row['renewables_share_elec'],
            'fossil_pct': row['fossil_share_elec'],
            'solar_pct': row['solar_share_elec'],
            'wind_pct': row['wind_share_elec'],
            'hydro_pct': row['hydro_share_elec'],
            'nuclear_pct': row['nuclear_share_elec'],
            'elec_gen': row['electricity_generation'],
            'coal_prod': row['coal_production'],
            'oil_prod': row['oil_production'],
            'gas_prod': row['gas_production'],
            'gdp_per_capita': row['gdp_per_capita'],
            'population': row['population']
        }

# Also build a flat list of available years and countries for the UI
meta_extra = {
    'years': sorted([int(y) for y in map_json.keys()]),
    'countries': sorted(map_df[['country','iso_code']].drop_duplicates()
                        .apply(lambda r: {'name': r['country'], 'iso': r['iso_code']}, axis=1)
                        .tolist(), key=lambda x: x['name'])
}

with open('../app/static/map_data.json', 'w') as f:
    json.dump(map_json, f, separators=(',', ':'))  # compact

with open('../app/static/map_meta.json', 'w') as f:
    json.dump(meta_extra, f, indent=2)

import os
size_kb = os.path.getsize('../app/static/map_data.json') / 1024
print(f"map_data.json: {size_kb:.0f} KB")
print(f"Years available: {meta_extra['years'][:5]} ... {meta_extra['years'][-3:]}")
print(f"Countries: {len(meta_extra['countries'])}")

## 2. Global Trend Lines

These time-series plots show global aggregates over the full 1990–2022 period.
They are embedded as static images in the map page sidebar.

In [ ]:
# Aggregate by year — use median to avoid China/US dominating the average
yearly = df.groupby('year').agg(
    total_energy=('primary_energy_consumption', 'sum'),
    median_renewables=('renewables_share_elec', 'median'),
    median_fossil=('fossil_share_elec', 'median'),
    median_energy_pc=('energy_per_capita', 'median'),
    n_countries=('country', 'count')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

NAVY = '#1a2e45'
GREEN = '#2d7a4f'
AMBER = '#d4830a'

axes[0].fill_between(yearly['year'], yearly['total_energy'] / 1000,
                     alpha=0.15, color=NAVY)
axes[0].plot(yearly['year'], yearly['total_energy'] / 1000,
             color=NAVY, linewidth=2)
axes[0].set_title('Global Total Energy (PWh)')
axes[0].set_ylabel('Petawatt-hours')
axes[0].set_xlabel('Year')

axes[1].plot(yearly['year'], yearly['median_renewables'],
             color=GREEN, linewidth=2, label='Renewables')
axes[1].plot(yearly['year'], yearly['median_fossil'],
             color=AMBER, linewidth=2, linestyle='--', label='Fossil')
axes[1].set_title('Median Electricity Mix (%)')
axes[1].set_ylabel('Share (%)')
axes[1].set_xlabel('Year')
axes[1].legend(frameon=False)

axes[2].fill_between(yearly['year'], yearly['median_energy_pc'],
                     alpha=0.15, color=AMBER)
axes[2].plot(yearly['year'], yearly['median_energy_pc'],
             color=AMBER, linewidth=2)
axes[2].set_title('Median Energy per Capita (MWh)')
axes[2].set_ylabel('MWh per person')
axes[2].set_xlabel('Year')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Global Energy Trends 1990–2022', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../app/static/map_global_trends.png', bbox_inches='tight')
plt.show()
print("Saved: app/static/map_global_trends.png")

## 3. Top/Bottom Country Tables (2021)

Pre-compute ranked tables for 2021 which the map page sidebar will display.

In [ ]:
yr2021 = df[df['year'] == 2021][[
    'country', 'iso_code', 'primary_energy_consumption',
    'energy_per_capita', 'renewables_share_elec', 'fossil_share_elec'
]].copy()

print("Top 10 by total energy consumption (2021):")
top10 = yr2021.nlargest(10, 'primary_energy_consumption')[['country','primary_energy_consumption']]
print(top10.to_string(index=False))

print("\nTop 10 by renewables share (2021, min 10 TWh consumption):")
top_ren = yr2021[yr2021['primary_energy_consumption'] > 10].nlargest(10, 'renewables_share_elec')[
    ['country','renewables_share_elec','primary_energy_consumption']]
print(top_ren.to_string(index=False))

# Save as JSON for sidebar use
rankings = {
    'top_energy': top10.rename(columns={'primary_energy_consumption':'value'}).to_dict('records'),
    'top_renewables': top_ren.rename(columns={'renewables_share_elec':'value'}).to_dict('records')
}
with open('../app/static/map_rankings.json', 'w') as f:
    json.dump(rankings, f, indent=2)
print("\nSaved: app/static/map_rankings.json")

## 4. Electricity Mix Breakdown — Regional Averages (2021)

Bar chart of electricity source breakdown for selected large economies, saved as a static image.

In [ ]:
HIGHLIGHT_COUNTRIES = [
    'United States', 'China', 'Germany', 'France',
    'India', 'Brazil', 'Norway', 'Australia'
]

mix_2021 = df[(df['year'] == 2021) & (df['country'].isin(HIGHLIGHT_COUNTRIES))][[
    'country', 'coal_production', 'oil_production', 'gas_production',
    'renewables_share_elec', 'fossil_share_elec',
    'solar_share_elec', 'wind_share_elec',
    'hydro_share_elec', 'nuclear_share_elec'
]].set_index('country')

# Stack bar of electricity shares
share_cols = ['solar_share_elec', 'wind_share_elec', 'hydro_share_elec',
              'nuclear_share_elec', 'fossil_share_elec']
share_labels = ['Solar', 'Wind', 'Hydro', 'Nuclear', 'Fossil']
share_colors = ['#f5c842', '#2d7a4f', '#2c85c1', '#8e44ad', '#c0392b']

present = [c for c in HIGHLIGHT_COUNTRIES if c in mix_2021.index]
plot_data = mix_2021.loc[present, share_cols]

fig, ax = plt.subplots(figsize=(13, 5))
bottom = np.zeros(len(present))

for col, label, color in zip(share_cols, share_labels, share_colors):
    vals = plot_data[col].values
    ax.bar(present, vals, bottom=bottom, label=label,
           color=color, edgecolor='white', linewidth=0.5)
    bottom += vals

ax.set_ylabel('Share of Electricity (%)')
ax.set_title('Electricity Generation Mix — Selected Countries (2021)')
ax.legend(loc='lower right', frameon=False, ncol=5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../app/static/map_elec_mix.png', bbox_inches='tight')
plt.show()
print("Saved: app/static/map_elec_mix.png")

## 5. Energy vs GDP per Capita Bubble Chart (2021)

Each country is a bubble — size = total energy, colour = renewables share.
Saved as static image for the map sidebar.

In [ ]:
bubble = df[df['year'] == 2021].copy()
bubble['gdp_pc'] = bubble['gdp'] / bubble['population']
bubble = bubble[bubble['gdp_pc'] > 0].copy()

# Normalise bubble size
sizes = (bubble['primary_energy_consumption'] /
         bubble['primary_energy_consumption'].max() * 800 + 10)

fig, ax = plt.subplots(figsize=(11, 6))

sc = ax.scatter(
    np.log10(bubble['gdp_pc']),
    bubble['energy_per_capita'],
    s=sizes,
    c=bubble['renewables_share_elec'],
    cmap='RdYlGn',
    alpha=0.65,
    edgecolors='white',
    linewidths=0.4
)

# Label a few notable countries
LABEL = ['United States', 'China', 'Norway', 'India', 'Germany', 'Brazil', 'Qatar']
for _, row in bubble[bubble['country'].isin(LABEL)].iterrows():
    ax.annotate(row['country'],
                (np.log10(row['gdp_pc']), row['energy_per_capita']),
                fontsize=8, color='#333',
                xytext=(4, 4), textcoords='offset points')

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Renewables Share (%)', fontsize=9)

ax.set_xlabel('log₁₀(GDP per Capita, USD)')
ax.set_ylabel('Energy per Capita (MWh/person)')
ax.set_title('Energy per Capita vs GDP per Capita (2021)\nBubble size = total energy consumption')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../app/static/map_bubble_2021.png', bbox_inches='tight')
plt.show()
print("Saved: app/static/map_bubble_2021.png")

## Done

All outputs saved to `app/static/`. The web app's `/map` route loads:
- `map_data.json` — for the interactive D3 choropleth
- `map_meta.json` — available years and country list
- `map_rankings.json` — pre-computed top-10 tables
- `map_global_trends.png` — time-series overview
- `map_elec_mix.png` — electricity mix stacked bar
- `map_bubble_2021.png` — GDP vs energy bubble chart